In [ ]:
import pandas as pd

df = pd.read_csv("Speed+Dating+Data.csv")

with pd.option_context("display.max_columns", None):
    display(df.head())

# Do people care about having the same interests?

In [ ]:
import pandas as pd
import plotly.express as px
from scipy import stats

# Filter for necessary columns and drop missing values
interests_df = df[['int_corr', 'match']].dropna()

# Split the dataset into two groups: Matches and Non-Matches
matched = interests_df[interests_df['match'] == 1]['int_corr']
not_matched = interests_df[interests_df['match'] == 0]['int_corr']

# Calculate group means for a quick check
print(f"Mean Interest Corr (Matches): {matched.mean():.4f}")
print(f"Mean Interest Corr (Non-Matches): {not_matched.mean():.4f}")

# Box Plot with Jittered Points
fig = px.box(interests_df, x='match', y='int_corr', 
             color='match',
             points="all", # Shows all individual data points next to the box
             title="Interest Correlation: Matches vs. Non-Matches",
             labels={'int_corr': 'Correlation of Interests (int_corr)', 'match': 'Did they Match?'},
             color_discrete_map={0: "lightgrey", 1: "seagreen"})

fig.update_layout(width=600, height=400)
fig.show()

# Independent Samples T-Test -> we test if the mean 'int_corr' of the matched group is different from the non-matched group
t_stat, p_value = stats.ttest_ind(matched, not_matched, equal_var=False)

print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {p_value:.4f}")

# Do people stay in their own ethnicity?

In [ ]:
import pandas as pd
import plotly.express as px
from scipy.stats import chi2_contingency

# Compare same-race pairings vs matching
homophily_df = df[['samerace', 'match']].dropna()

# Calculate match rates
rate_table = homophily_df.groupby('samerace')['match'].value_counts(normalize=True).unstack()
rate_table.index = ['Different Race', 'Same Race']
rate_table.columns = ['No Match', 'Match']

# Comparison bar chart
fig = px.bar(rate_table, y='Match', 
             title="Match Rate: Same Race vs. Different Race",
             labels={'index': 'Pairing Type', 'Match': 'Percentage of Matches'},
             color=rate_table.index, color_discrete_sequence=px.colors.qualitative.Pastel)
fig.show()

# Chi-Square Test of Independence -> test if 'match' and 'samerace' are independent variables
contingency_table = pd.crosstab(homophily_df['samerace'], homophily_df['match']) # creates frequency table from index/columns
chi2, p, dof, ex = chi2_contingency(contingency_table)

print(f"Chi-Square Statistic: {chi2:.4f}")
print(f"P-value: {p:.4f}")

# What is the influence of someone's career?

In [ ]:
import pandas as pd
import plotly.express as px
from scipy.stats import f_oneway

# Mapping field codes to names
# field_cd mapping from dataset documentation
field_map = {
    1: 'Law', 2: 'Math', 3: 'Soc. Science', 4: 'Med. Science', 5: 'Eng', 
    6: 'Journ', 7: 'Hist', 8: 'Bus', 9: 'Edu', 10: 'Bio', 11: 'Soc. Work', 
    12: 'Undergrad', 13: 'Pol. Sci', 14: 'Film', 15: 'Arts', 16: 'Lang', 
    17: 'Arch', 18: 'Other'
}

career_df = df[['field_cd', 'like_o']].dropna()
career_df['field_name'] = career_df['field_cd'].map(field_map)

# Calculate mean likability per field
field_stats = career_df.groupby('field_name')['like_o'].mean().sort_values(ascending=False).reset_index()

# 2. Plotly: Ranking Likability by Field
fig = px.bar(field_stats, x='like_o', y='field_name', orientation='h',
             title="Average Likability Rating Received by Field of Study",
             labels={'like_o': 'Avg Rating from Dates (1-10)', 'field_name': 'Field'},
             color='like_o', color_continuous_scale='Viridis')
fig.update_layout(yaxis={'categoryorder':'total ascending'})
fig.show()

# One-Way ANOVA -> test if the differences in the mean likability scores between fields are significant
groups = [group['like_o'].values for name, group in career_df.groupby('field_name')]
f_stat, p_value = f_oneway(*groups)

print(f"ANOVA F-statistic: {f_stat:.4f}")
print(f"ANOVA P-value: {p_value:.4f}")